# Softmax 역확산 보정 v2 — Drive 마운트 / 네 칸

L4에서 1→2→3번 실행합니다. 1번에서 학습 자료가 있는 계정으로 Google Drive 연결을 승인하세요.
기존 Drive의 ZIP 또는 원래 생성 실험 폴더와 당시 획 추출기 best.pt를 사용합니다.
기본은 가·걔·곁·괴 네 칸 모두입니다. 기준 획 조건을 사용하는 추가 잉크 삭제 실험입니다.
새 첫 생성과 Qwen 로딩은 없습니다. 기존 오류 이미지를 추가 역확산합니다.
계산과 중간 저장은 /content에서 수행하고 최종 ZIP은 Drive에 복사합니다.
Google 인증/네트워크 오류 자체를 해결했다고 보장하지 않으며 마운트 실패 시 원래 오류와 함께 중단합니다.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess,sys,tempfile,json,hashlib,gc
from google.colab import drive
import os
FORCE_REMOUNT=False  # 연결이 깨진 기존 mount를 다시 연결할 때만 True
if FORCE_REMOUNT or not os.path.ismount('/content/drive'):
    print('Google Drive 연결을 승인하세요. 기존 학습 자료가 있는 계정을 선택하세요.',flush=True)
    try:
        drive.mount('/content/drive',force_remount=FORCE_REMOUNT,timeout_ms=180000)
    except Exception as exc:
        raise RuntimeError('Drive 인증/마운트에 실패했습니다. 위 Google 오류를 확인한 뒤 1번 셀을 다시 실행하세요.') from exc
print('Drive mount 확인 완료',flush=True)
root=Path(tempfile.mkdtemp(prefix='saved_repair_',dir='/content'))
subprocess.run(['git','clone','--depth','1','https://github.com/yechan25/hsqm.git',str(root)],check=True)
commit=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip()
subprocess.run([sys.executable,'-m','pip','install','--prefer-binary',
    'diffusers==0.37.0','transformers==4.57.6','huggingface-hub==0.36.0',
    'accelerate','safetensors','timm','pandas','numpy','matplotlib','pillow',
    'scipy','scikit-image','pyyaml','fonttools','ipywidgets'],check=True)
runtime_helpers={'diffusion_model/saved_repair_demo.py': '"""Load an actual saved baseline and its recorded reference conditions, not a new generation."""\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nimport zipfile\nimport numpy as np\nfrom PIL import Image\n\n\ndef pack_saved_generation(report_path, output):\n    """Package an existing Drive experiment without its original dataset/CSV."""\n    from stroke_model.convert_reference_csv import existing_path\n    report_path=existing_path(report_path)\n    report=json.loads(report_path.read_text(encoding=\'utf-8\'))\n    output=Path(output)\n    if output.exists():raise FileExistsError(output)\n    pages=[p for p in report[\'pages\'] if p[\'method\']==\'baseline\']\n    rows=[r for r in report[\'results\'] if r[\'method\']==\'reference\']\n    if len(pages)!=1 or len(rows)!=4:raise ValueError(\'Expected one baseline and four recorded reference slots\')\n    def locate(recorded):\n        basename=PurePosixPath(str(recorded).replace(\'\\\\\',\'/\')).name\n        local=report_path.parent/basename\n        return existing_path(local if local.is_file() else recorded)\n    source=locate(pages[0][\'input\'])\n    references=[locate(r[\'npz\']) for r in rows]\n    output.parent.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(output,\'w\',zipfile.ZIP_DEFLATED) as z:\n        z.write(source,\'generation/\'+source.name)\n        for path in references:z.write(path,\'generation/\'+path.name)\n        z.writestr(\'generation/generation_report.json\',json.dumps(report,ensure_ascii=False))\n    return output\n\n\ndef prepare_saved_demo(archive, output):\n    destination=Path(output)\n    if destination.exists():raise FileExistsError(destination)\n    with zipfile.ZipFile(archive) as z:\n        names=z.namelist()\n        reports=[n for n in names if n.endswith(\'generation_report.json\')]\n        if len(reports)!=1:raise ValueError(\'ZIP must contain exactly one generation_report.json\')\n        report=json.loads(z.read(reports[0]))\n        def member(recorded):\n            basename=PurePosixPath(str(recorded).replace(\'\\\\\',\'/\')).name\n            matches=[n for n in names if PurePosixPath(n).name==basename]\n            if len(matches)!=1:raise ValueError(f\'Missing/ambiguous ZIP file: {basename}\')\n            return matches[0]\n        baseline=[p for p in report[\'pages\'] if p[\'method\']==\'baseline\']\n        if len(baseline)!=1:raise ValueError(\'A single recorded baseline page is required\')\n        source_member=member(baseline[0][\'input\'])\n        source_bytes=z.read(source_member)\n        with Image.open(io.BytesIO(source_bytes)) as image:\n            width,height=image.size\n        if width!=height or width%8:raise ValueError(\'Expected a square saved sheet, size divisible by 8\')\n        # Reference arrays are metadata conditions AFTER generation, never the repair input.\n        templates=[];boxes=[]\n        for index in range(4):\n            candidates=[r for r in report[\'results\'] if r[\'method\']==\'reference\' and r[\'slot\']==index]\n            if len(candidates)!=1:raise ValueError(\'Four reference slots required\')\n            row=candidates[0];boxes.append(row[\'crop_bbox\'])\n            try:npz_member=member(row[\'npz\'])\n            except ValueError:npz_member=member(f\'reference_slot_{index+1}.npz\')\n            with np.load(io.BytesIO(z.read(npz_member)),allow_pickle=False) as data:\n                ref=data[\'reference\'].astype(np.float32)\n                strokes=data[\'reference_strokes\'].astype(np.float32)\n            if ref.ndim!=3 or strokes.ndim!=3 or ref.shape[0]!=1 or ref.shape[1:]!=strokes.shape[1:]:\n                raise ValueError(\'Malformed saved reference conditions\')\n            templates.append((ref,strokes))\n    destination.mkdir(parents=True)\n    source=destination/\'saved_error_input.png\';source.write_bytes(source_bytes)\n    manifest=[]\n    for i,(reference,strokes) in enumerate(templates):\n        name=f\'condition_{i+1}.npz\'\n        np.savez_compressed(destination/name,reference=reference,strokes=strokes)\n        manifest.append(dict(path=name,split=\'train\',char_id=\'\'))\n    path=destination/\'conditions.json\';path.write_text(json.dumps(manifest),encoding=\'utf-8\')\n    provenance=dict(source_archive=str(archive),source_member=source_member,\n        source_sha256=hashlib.sha256(source_bytes).hexdigest(),\n        source_initialization=report.get(\'init_mode\',\'not_recorded_in_legacy_report\'),\n        source_prompt=report.get(\'prompt\'),source_report=report,\n        demonstration=\'reference_conditioned_repair_of_saved_baseline\',\n        fresh_text_to_image=False,reference_encoded_as_repair_input=False)\n    (destination/\'source_provenance.json\').write_text(json.dumps(provenance,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    return dict(source=str(source),manifest=str(path),rows=[1,2,3,4],boxes=boxes,\n        size=width,checkpoint=report[\'checkpoint\'],checkpoint_sha256=report.get(\'checkpoint_sha256\'),\n        provenance=provenance)\n', 'stroke_model/saved_extra_guide.py': '"""Reference-conditioned extra ink from frozen extractor ownership.\n\nThis is an aligned-reference demonstration, not reference-free error detection.\nSupported reference strokes are protected. For partition extractors, summing\noutside all stroke supports reduces to a geometric spill map. The notebook\nchecks this limitation explicitly rather than attributing that effect to learning.\n"""\nimport torch\nfrom torch.nn import functional as F\n\n\n@torch.no_grad()\ndef extracted_extra_map(model,rgb,slots,*,focus_slots=(0,),tolerance=2):\n    if type(tolerance) is not int or tolerance<0:raise ValueError(\'Nonnegative integer tolerance required\')\n    if not focus_slots or any(i not in range(len(slots)) for i in focus_slots):raise ValueError(\'Invalid focus slots\')\n    probability=torch.zeros_like(rgb[:,:1]);geometry=torch.zeros_like(probability)\n    details=[]\n    for i,slot in enumerate(slots):\n        x0,y0,x1,y1=slot[\'bbox\']\n        reference=torch.as_tensor(slot[\'reference\'],device=rgb.device,dtype=torch.float32)[None]\n        strokes=torch.as_tensor(slot[\'strokes\'],device=rgb.device,dtype=torch.float32)[None]\n        crop=F.interpolate(rgb[...,y0:y1,x0:x1],reference.shape[-2:],mode=\'bilinear\',align_corners=False)\n        ink=1-(crop*crop.new_tensor([.299,.587,.114])[None,:,None,None]).sum(1,keepdim=True)\n        prediction=model(ink,reference,strokes)\n        masks=prediction[\'masks\']\n        support=F.max_pool2d(strokes,2*tolerance+1,1,tolerance)\n        assignment_extra=(masks*(1-support)).sum(1,keepdim=True).clamp(0,1)\n        # A different stroke may legitimately own ink here. Never erase inside\n        # ANY supported reference stroke merely because assignment differs.\n        outside=(support.amax(1,keepdim=True)<.1).float()\n        extra=assignment_extra*outside\n        union=ink*outside\n        if i in focus_slots:\n            probability[...,y0:y1,x0:x1]=F.interpolate(extra,(y1-y0,x1-x0),mode=\'bilinear\',align_corners=False)\n            geometry[...,y0:y1,x0:x1]=F.interpolate(union,(y1-y0,x1-x0),mode=\'bilinear\',align_corners=False)\n        details.append(dict(slot=i,bbox=list(slot[\'bbox\']),masks=masks[0].cpu().numpy(),\n            reference_strokes=strokes[0].cpu().numpy(),extra=extra[0,0].cpu().numpy(),\n            assignment_extra=assignment_extra[0,0].cpu().numpy()))\n    return probability,geometry,details\n\n\ndef fixed_region_metrics(rgb,objective):\n    """Optimization-region diagnostics, not an independent quality score."""\n    with torch.no_grad():\n        error=(rgb-objective.target).abs().mean(1,keepdim=True)\n        mask=objective.mask\n        outside=1-mask\n        return dict(edit_region_mae=float((error*mask).sum()/mask.sum().clamp_min(1)),\n            protected_region_mae=float(((rgb-objective.source).abs().mean(1,keepdim=True)*outside).sum()/outside.sum().clamp_min(1)),\n            protected_peak_channel_change=float(((rgb-objective.source).abs()*outside).amax()),\n            edit_pixels=int(mask.sum()))\n', 'diffusion_model/guided_ddim.py': '"""Backward clean-latent guidance for a frozen epsilon DDIM model.\n\nGradient is through VAE -> frozen Topo -> soft stroke loss, NOT through U-Net.\nEither convert a clean-latent correction back to epsilon (legacy), or apply an\nexplicit guided mean update with the U-Net residual held fixed. Backtrack until\nthe SAME timestep\'s objective decreases. This does not guarantee final quality.\n"""\nfrom copy import deepcopy\nfrom collections import Counter\nfrom dataclasses import asdict, dataclass\nimport math\n\nimport torch\nfrom stroke_model.source_ink_guard import local_ink_change\n\n\n@dataclass(frozen=True)\nclass GuidanceConfig:\n    noise_ratio: float=.05\n    clean_step_ratio: float | None=None\n    start_fraction: float=.30\n    end_fraction: float=.90\n    every: int=2\n    backtracks: int=2\n    snapshot_every: int=4\n    healthy_stroke_score: float=90.\n    max_healthy_stroke_drop: float=1.\n    state_update: str=\'epsilon\'\n    max_new_local_ink: float | None=None\n    local_edit: bool=False\n    local_iterations: int=24\n    local_halo_radius: int=0\n    local_final_iterations: int=0\n    local_erase_l1_weight: float=0.\n    local_erase_tail_weight: float=0.\n    max_state_change_to_base_step_ratio: float | None=None\n    inner_iterations: int=1\n    final_inner_iterations: int=1\n    max_clean_change_ratio: float=.08\n\n    def __post_init__(self):\n        if (type(self.inner_iterations) is not int or self.inner_iterations<1 or\n                type(self.final_inner_iterations) is not int or self.final_inner_iterations<1 or\n                not math.isfinite(self.max_clean_change_ratio) or self.max_clean_change_ratio<=0):\n            raise ValueError(\'Invalid iterative guidance budget\')\n        if (self.max_state_change_to_base_step_ratio is not None and\n                (not math.isfinite(self.max_state_change_to_base_step_ratio) or\n                 self.max_state_change_to_base_step_ratio<=0 or self.state_update!=\'clean_residual\' or self.local_edit)):\n            raise ValueError(\'State change cap requires nonlocal clean_residual guidance and a positive ratio\')\n        if (not math.isfinite(self.local_erase_tail_weight) or self.local_erase_tail_weight<0\n                or not math.isfinite(self.local_erase_l1_weight) or self.local_erase_l1_weight<0\n                or type(self.local_halo_radius) is not int or self.local_halo_radius<0\n                or type(self.local_final_iterations) is not int or self.local_final_iterations<0\n                or type(self.local_iterations) is not int or self.local_iterations<1\n                or (self.local_edit and self.state_update!=\'clean_residual\')\n                or not math.isfinite(self.noise_ratio) or self.noise_ratio<0\n                or (self.clean_step_ratio is not None and\n                    (not math.isfinite(self.clean_step_ratio) or self.clean_step_ratio<0))\n                or not 0<=self.start_fraction<self.end_fraction<=1\n                or type(self.every) is not int or self.every<1\n                or type(self.backtracks) is not int or self.backtracks<0\n                or type(self.snapshot_every) is not int or self.snapshot_every<1\n                or not math.isfinite(self.healthy_stroke_score) or not 0<=self.healthy_stroke_score<=100\n                or not math.isfinite(self.max_healthy_stroke_drop) or self.max_healthy_stroke_drop<0\n                or self.state_update not in (\'epsilon\',\'clean_residual\')\n                or (self.state_update==\'clean_residual\' and self.clean_step_ratio is None)\n                or (self.max_new_local_ink is not None and\n                    (not math.isfinite(self.max_new_local_ink) or self.max_new_local_ink<0))):\n            raise ValueError(\'Invalid guidance schedule\')\n\n\ndef decode_estimate(pipe,z0):\n    dtype=next(pipe.vae.parameters()).dtype\n    decoded=pipe.vae.decode(z0.to(dtype)/pipe.vae.config.scaling_factor,return_dict=False)[0]\n    return decoded.float()/2+.5\n\n\ndef _plain(value):\n    if torch.is_tensor(value):return value.detach().cpu().tolist()\n    if isinstance(value,dict):return {key:_plain(item) for key,item in value.items()}\n    if isinstance(value,(list,tuple)):return [_plain(item) for item in value]\n    return value\n\n\ndef _scalar_rows(details):\n    return _plain(details[\'per_glyph\'])\n\n\ndef _healthy_guard(before,after,config):\n    """Protect EACH previously healthy stroke, independent of sheet averages."""\n    drops=[]\n    for a,b in zip(before,after):\n        sa,sb=a.get(\'per_stroke\',[]),b.get(\'per_stroke\',[])\n        if len(sa)!=len(sb):raise ValueError(\'Candidate stroke identities changed\')\n        for first,second in zip(sa,sb):\n            if first[\'stroke\']!=second[\'stroke\']:raise ValueError(\'Candidate stroke order changed\')\n            if first[\'score\']>=config.healthy_stroke_score:\n                drops.append(float(first[\'score\']-second[\'score\']))\n    drop=max([0.]+drops)\n    return drop<=config.max_healthy_stroke_drop+1e-5,drop\n\n\ndef _mean_score(details):\n    key=\'extracted_stroke_score_no_extra\'\n    if not all(key in row for row in details[\'per_glyph\']):return None\n    return sum(float(row[key].detach()) for row in details[\'per_glyph\'])/len(details[\'per_glyph\'])\n\n\ndef format_guidance_record(record):\n    """Do not conflate scheduled-off guidance with rejected proposals."""\n    active=bool(record.get(\'active\'));accepted=bool(record.get(\'accepted\'))\n    state=\'accepted\' if accepted else \'rejected\' if active else \'inactive\'\n    fields=[f\'active={active}\',f\'accepted={accepted}\',f\'state={state}\']\n    if active:\n        fields.append(f"inner={record.get(\'accepted_inner_steps\',int(accepted))}")\n        fields.append(f"edit={record.get(\'edit_loss_before\',record.get(\'loss_before\'))} -> {record.get(\'edit_loss_after\',record.get(\'loss_after\'))}")\n        reasons=Counter(a[\'reason\'] for a in record.get(\'attempts\',[]) if a.get(\'reason\')!=\'accepted\')\n        if reasons:fields.append(\'reject=\'+\',\'.join(f\'{name}:{count}\' for name,count in reasons.items()))\n        failed=Counter(f[\'name\'] for a in record.get(\'attempts\',[]) for f in a.get(\'guard_failures\',[]))\n        if failed:fields.append(\'guards=\'+\',\'.join(f\'{name}:{count}\' for name,count in failed.items()))\n        if record.get(\'anchor_guard_failures\'):fields.append(\'anchor_already_outside_guard=True\')\n    return \' | \'.join(fields)\n\n\ndef denoise(pipe,initial,timesteps,embeddings,*,objective=None,config=None,cfg_scale=7.5,on_step=None):\n    config=config or GuidanceConfig(noise_ratio=0)\n    enabled=(config.clean_step_ratio>0 if config.clean_step_ratio is not None else config.noise_ratio>0)\n    if enabled and objective is None:\n        raise ValueError(\'An actual differentiable Topo objective is required for guidance\')\n    scheduler=deepcopy(pipe.scheduler)\n    if scheduler.config.prediction_type!=\'epsilon\':raise ValueError(\'Epsilon prediction required\')\n    if scheduler.config.clip_sample or getattr(scheduler.config,\'thresholding\',False):\n        raise ValueError(\'This clean-latent correction requires clip_sample=False and thresholding=False\')\n    # Keep small DDIM state updates in fp32. Only U-Net inputs use model dtype.\n    latent=initial.detach().float().clone()\n    unet_dtype=next(pipe.unet.parameters()).dtype\n    trace=[];ink_anchor=None;editor=None\n    if not enabled and hasattr(objective,\'begin_baseline\'):objective.begin_baseline()\n    for index,t in enumerate(timesteps):\n        with torch.no_grad():\n            inputs=torch.cat([latent,latent]) if cfg_scale>1 else latent\n            inputs=scheduler.scale_model_input(inputs,t)\n            predicted=pipe.unet(inputs.to(unet_dtype),t,encoder_hidden_states=embeddings,return_dict=False)[0]\n            if cfg_scale>1:\n                uncond,text=predicted.chunk(2)\n                predicted=uncond+cfg_scale*(text-uncond)\n            predicted=predicted.float()\n            original=scheduler.step(predicted,t,latent,eta=0.,return_dict=True)\n            if not enabled and hasattr(objective,\'record_baseline\'):\n                objective.record_baseline(index,decode_estimate(pipe,original.pred_original_sample))\n        progress=index/max(len(timesteps)-1,1)\n        active=(objective is not None and enabled and index%config.every==0\n                and config.start_fraction<progress<=config.end_fraction)\n        record=dict(step=index,timestep=int(t),active=active,accepted=False,state_update=config.state_update,\n                    base_transition_rms=float((original.prev_sample-latent).square().mean().sqrt()))\n        chosen=original\n        if active and hasattr(objective,\'optimize_clean\'):\n            if config.state_update!=\'clean_residual\' or config.clean_step_ratio is None:\n                raise ValueError(\'Iterative objective requires clean_residual guidance\')\n            edited,summary=objective.optimize_clean(original.pred_original_sample,\n                lambda z:decode_estimate(pipe,z),iterations=(max(config.inner_iterations,config.final_inner_iterations)\n                    if index==len(timesteps)-1 else config.inner_iterations),\n                step_ratio=config.clean_step_ratio,backtracks=config.backtracks,\n                budget_ratio=config.max_clean_change_ratio,step_index=index)\n            record.update(summary)\n            if summary[\'accepted\']:\n                delta=edited-original.pred_original_sample\n                prev_t=int(t)-scheduler.config.num_train_timesteps//scheduler.num_inference_steps\n                a=(scheduler.alphas_cumprod[prev_t] if prev_t>=0 else scheduler.final_alpha_cumprod).to(latent)\n                chosen=type(original)(prev_sample=original.prev_sample+a.sqrt()*delta,pred_original_sample=edited)\n                record[\'state_delta_rms\']=float((chosen.prev_sample-original.prev_sample).square().mean().sqrt())\n                record[\'clean_delta_rms\']=float(delta.square().mean().sqrt())\n        elif active and config.local_edit:\n            from stroke_model.local_reference_edit import LocalReferenceEdit,optimize_local\n            if objective.source_guard is None:raise ValueError(\'Local edit requires reference sheet\')\n            if editor is None:\n                with torch.no_grad():\n                    anchor=decode_estimate(pipe,original.pred_original_sample)\n                    reference=(1-objective.source_guard.reference_ink).expand_as(anchor)\n                    editor=LocalReferenceEdit(reference,anchor,halo_radius=config.local_halo_radius,\n                        erase_l1_weight=config.local_erase_l1_weight,\n                        erase_tail_weight=config.local_erase_tail_weight)\n                    objective.local_editor=editor\n            with torch.no_grad():editor.rebase(decode_estimate(pipe,original.pred_original_sample))\n            edited,summary=optimize_local(original.pred_original_sample,\n                lambda z:decode_estimate(pipe,z),editor,iterations=(max(config.local_iterations,config.local_final_iterations)\n                    if index==len(timesteps)-1 else config.local_iterations),\n                step_ratio=config.clean_step_ratio)\n            record.update(summary)\n            if summary[\'accepted\']:\n                delta=edited-original.pred_original_sample\n                prev_t=int(t)-scheduler.config.num_train_timesteps//scheduler.num_inference_steps\n                a=(scheduler.alphas_cumprod[prev_t] if prev_t>=0 else scheduler.final_alpha_cumprod).to(latent)\n                chosen=type(original)(prev_sample=original.prev_sample+a.sqrt()*delta,\n                                      pred_original_sample=edited)\n                record[\'state_delta_rms\']=float((chosen.prev_sample-original.prev_sample).square().mean().sqrt())\n        elif active:\n            # Single-step leaf: never retain the entire denoising trajectory.\n            with torch.enable_grad():\n                clean=original.pred_original_sample.detach().float().requires_grad_(True)\n                loss,details=objective(decode_estimate(pipe,clean))\n                if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite Topo guidance loss\')\n                gradient,=torch.autograd.grad(loss,clean)\n            if not torch.isfinite(gradient).all():raise FloatingPointError(\'Nonfinite clean-latent gradient\')\n            norm=gradient.square().mean().sqrt()\n            record.update(loss_before=float(loss.detach()),gradient_rms=float(norm),\n                topo_score_before=_mean_score(details),per_glyph_before=_scalar_rows(details),attempts=[])\n            priorities=objective.freeze_priority(details) if hasattr(objective,\'freeze_priority\') else None\n            source_before=details.get(\'source_spill_map\')\n            if config.max_new_local_ink is not None and source_before is None:\n                raise ValueError(\'Local ink guard requires the fixed native reference sheet\')\n            if details.get(\'source_spill_loss\') is not None:\n                record[\'source_spill_loss_before\']=float(details[\'source_spill_loss\'].detach())\n            del loss,details,clean\n            if norm>1e-12:\n                if config.clean_step_ratio is not None:\n                    # Specify a clean-latent step, then convert it EXACTLY to epsilon.\n                    # Ramp in, but keep final-step correction instead of tapering to zero.\n                    schedule=min(1.,(progress-config.start_fraction)/max(\n                        .25*(config.end_fraction-config.start_fraction),1e-6))\n                    alpha=scheduler.alphas_cumprod[int(t)].to(latent.device).float()\n                    clean_rms=original.pred_original_sample.float().square().mean().sqrt().clamp_min(1e-3)\n                    desired=config.clean_step_ratio*schedule*clean_rms*gradient/norm\n                    correction=(alpha/(1-alpha).clamp_min(1e-12)).sqrt()*desired\n                    record[\'correction_budget\']=\'clean_latent_rms\'\n                else:\n                    schedule=math.sin(math.pi*(progress-config.start_fraction)/(config.end_fraction-config.start_fraction))\n                    rms=predicted.float().square().mean().sqrt().clamp_min(1e-3)\n                    correction=config.noise_ratio*schedule*rms*gradient/norm\n                    record[\'correction_budget\']=\'epsilon_rms\'\n                with torch.no_grad():\n                    for attempt in range(config.backtracks+1):\n                        candidate_noise=(predicted.float()+correction*(.5**attempt)).to(predicted.dtype)\n                        actual_delta=candidate_noise.float()-predicted.float()\n                        if config.state_update==\'clean_residual\':\n                            # Keep the U-Net residual direction FIXED. Modifying both\n                            # x0 and epsilon in ordinary DDIM largely cancels the step.\n                            # This is an explicit guided-mean update, not an ordinary\n                            # DDIM step from a corrected epsilon at fixed z_t.\n                            delta=desired*(.5**attempt)\n                            prev_t=int(t)-scheduler.config.num_train_timesteps//scheduler.num_inference_steps\n                            prev_alpha=(scheduler.alphas_cumprod[prev_t] if prev_t>=0\n                                        else scheduler.final_alpha_cumprod).to(latent.device).float()\n                            if config.max_state_change_to_base_step_ratio is not None:\n                                cap=config.max_state_change_to_base_step_ratio*record[\'base_transition_rms\']\n                                rms=prev_alpha.sqrt()*delta.square().mean().sqrt()\n                                delta=delta*(cap/rms.clamp_min(1e-12)).clamp(max=1.)\n                            candidate=type(original)(\n                                prev_sample=original.prev_sample-prev_alpha.sqrt()*delta,\n                                pred_original_sample=original.pred_original_sample-delta)\n                        else:\n                            candidate=scheduler.step(candidate_noise,t,latent,eta=0.,return_dict=True)\n                        image=decode_estimate(pipe,candidate.pred_original_sample.float())\n                        after,after_details=(objective(image,stroke_weights=priorities) if priorities is not None\n                                             else objective(image))\n                        if not torch.isfinite(after):raise FloatingPointError(\'Nonfinite candidate loss\')\n                        after_score=_mean_score(after_details)\n                        score_ok=(record[\'topo_score_before\'] is None or after_score>=record[\'topo_score_before\']-1e-5)\n                        healthy_ok,healthy_drop=_healthy_guard(record[\'per_glyph_before\'],\n                            _scalar_rows(after_details),config)\n                        ink_ok=True;ink_increase=ink_violation=0.\n                        if config.max_new_local_ink is not None:\n                            if after_details.get(\'source_spill_map\') is None:\n                                raise ValueError(\'Candidate lost its source ink guard\')\n                            ink_ok,ink_increase,ink_violation,ink_anchor=local_ink_change(\n                                source_before,after_details[\'source_spill_map\'],ink_anchor,config.max_new_local_ink)\n                        state_delta=float((candidate.prev_sample-original.prev_sample).square().mean().sqrt())\n                        clean_delta=float((candidate.pred_original_sample-original.pred_original_sample).square().mean().sqrt())\n                        reason=(\'loss_not_lower\' if float(after)>=record[\'loss_before\'] else\n                                \'primary_score_lower\' if not score_ok else\n                                \'healthy_stroke_worse\' if not healthy_ok else\n                                \'new_local_ink\' if not ink_ok else\n                                \'no_state_change\' if state_delta==0 else \'accepted\')\n                        record[\'attempts\'].append(dict(attempt=attempt,loss_after=float(after),\n                            topo_score_after=after_score,state_delta_rms=state_delta,\n                            clean_delta_rms=clean_delta,max_healthy_stroke_drop=healthy_drop,reason=reason,\n                            new_local_ink_increase=ink_increase,new_local_ink_violation=ink_violation,\n                            state_to_clean_transfer_ratio=state_delta/max(clean_delta,1e-12)))\n                        if reason==\'accepted\':\n                            chosen=candidate\n                            record.update(accepted=True,backtracks=attempt,loss_after=float(after),\n                                correction_rms=float(actual_delta.square().mean().sqrt()),\n                                topo_score_after=after_score,per_glyph_after=_scalar_rows(after_details),\n                                state_delta_rms=state_delta,clean_delta_rms=clean_delta,\n                                state_to_clean_transfer_ratio=state_delta/max(clean_delta,1e-12),\n                                state_change_to_base_step_ratio=state_delta/max(record[\'base_transition_rms\'],1e-12))\n                            if after_details.get(\'source_spill_loss\') is not None:\n                                record[\'source_spill_loss_after\']=float(after_details[\'source_spill_loss\'])\n                            if ink_anchor is not None:\n                                ink_anchor=torch.minimum(ink_anchor,after_details[\'source_spill_map\'])\n                            break\n            if not record[\'accepted\']:record[\'loss_after\']=record[\'loss_before\']\n        latent=chosen.prev_sample.detach()\n        trace.append(record)\n        if on_step:\n            on_step(record,original.pred_original_sample.detach(),chosen.pred_original_sample.detach())\n        del original,chosen,predicted\n    return latent,trace\n', 'diffusion_model/run_topo_guidance.py': '"""Paired four-glyph SD1.5 epsilon/DDIM experiment, fixed .65 / seed 100."""\nfrom dataclasses import asdict\nfrom datetime import datetime\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nimport torch\n\nfrom .guided_ddim import GuidanceConfig, decode_estimate, denoise, format_guidance_record\nfrom .multiglyph_pilot import compose_sheet\nfrom .score_pilot import scored_ink, training_catalog\nfrom stroke_model.diffusion_guidance import TopoGuidance\nfrom stroke_model.infer_topology import load_topology, export_hard_masks\nfrom stroke_model.stroke_structure import StructureConfig\nfrom stroke_model.stroke_appearance import AppearanceConfig\n\n\ndef tensor_hash(tensor):\n    return hashlib.sha256(tensor.detach().contiguous().cpu().numpy().tobytes()).hexdigest()\n\n\ndef _file_hash(path):\n    digest=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda:stream.read(1024*1024),b\'\'):digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef generate_guided_pair(manifest,manifest_rows,checkpoint,output,*,model_id=\n        \'stable-diffusion-v1-5/stable-diffusion-v1-5\',revision=None,steps=30,canvas_size=576,\n        guidance=None,structure_config=None,extra_weight=.05,stroke_focus=None,compare_uniform=False,\n        appearance_weight=.5,appearance_config=None,compare_structure=True,\n        appearance_target=\'extractor_reference\',mask_gradient=\'ink_only\',source_spill_weight=.5,direct_reference=False,\n        guidance_signal="pixel_control",init_mode="reference",input_sheet=None,prompt_text=None,seed=100,strength=.65,baseline_only=False,preservation_weight=0.,objective_override=None,slot_bboxes=None):\n    if not torch.cuda.is_available():raise RuntimeError(\'Run this paired experiment on a Colab GPU\')\n    if not np.isfinite(strength) or not 0<strength<=1:raise ValueError(\'strength must be in (0,1]\')\n    if not np.isfinite(preservation_weight) or preservation_weight<0:raise ValueError(\'Invalid preservation weight\')\n    if preservation_weight and (guidance_signal!=\'template_structure\' or init_mode!=\'external\'):\n        raise ValueError(\'Preservation requires template structure guidance from an external generated image\')\n    if type(seed) is not int or seed<0:raise ValueError(\'Nonnegative integer seed required\')\n    if guidance_signal not in (\'pixel_control\',\'template_structure\',\'localized_extra\'):raise ValueError(\'Unknown guidance signal\')\n    if guidance_signal==\'localized_extra\' and (objective_override is None or init_mode!=\'external\'):\n        raise ValueError(\'Localized repair requires an external generated image and fixed error-map objective\')\n    if init_mode not in (\'reference\',\'external\',\'noise\'):raise ValueError(\'Unknown initialization mode\')\n    if init_mode==\'noise\' and not prompt_text:raise ValueError(\'Explicit target prompt required for noise generation\')\n    if init_mode==\'external\' and input_sheet is None:raise ValueError(\'External input sheet required\')\n    selected=list(manifest_rows)\n    catalog={row[\'manifest_row\']:row for row in training_catalog(manifest)}\n    if len(selected)!=4 or len(set(selected))!=4 or any(row not in catalog for row in selected):\n        raise ValueError(\'Select exactly four distinct training-reference rows\')\n    if type(steps) is not int or steps<4:raise ValueError(\'Need at least four nominal steps\')\n    if init_mode!=\'noise\' and int(steps*strength)<2:raise ValueError(\'Need at least two actual img2img timesteps\')\n    config=(GuidanceConfig(**guidance) if isinstance(guidance,dict) else guidance or\n            GuidanceConfig(clean_step_ratio=.01,end_fraction=1.,every=1,backtracks=5,\n                           state_update=\'clean_residual\',max_new_local_ink=.002))\n    if guidance_signal in (\'template_structure\',\'localized_extra\') and (config.local_edit or config.max_new_local_ink is not None):\n        raise ValueError(\'Template structure mode requires local_edit=False and max_new_local_ink=None\')\n    if guidance_signal in (\'template_structure\',\'localized_extra\') and (compare_uniform or compare_structure):\n        raise ValueError(\'Use only baseline/guided in template structure mode\')\n    if appearance_target not in (\'extractor_reference\',\'partition_reference\'):\n        raise ValueError(\'Unknown appearance target mode\')\n    structure=StructureConfig(**structure_config) if isinstance(structure_config,dict) else structure_config or StructureConfig()\n    appearance=(AppearanceConfig(**appearance_config) if isinstance(appearance_config,dict)\n                else appearance_config or AppearanceConfig())\n    focus=dict(focus_strength=2.,max_stroke_weight=3.)\n    if stroke_focus is not None:focus.update(stroke_focus)\n    initial_image,slots=compose_sheet([catalog[row] for row in selected],canvas_size)\n    if slot_bboxes is not None:\n        if len(slot_bboxes)!=len(slots):raise ValueError(\'One explicit bbox per intended character required\')\n        for slot,box in zip(slots,slot_bboxes):\n            if len(box)!=4 or any(type(v) is not int for v in box):raise ValueError(\'Integer bboxes required\')\n            x0,y0,x1,y1=box\n            if not (0<=x0<x1<=canvas_size and 0<=y0<y1<=canvas_size):raise ValueError(\'BBox outside generated image\')\n            slot[\'bbox\']=list(box)\n    destination=Path(output)\n    if destination.exists():raise FileExistsError(destination)\n    model,size,epoch=load_topology(checkpoint,torch.device(\'cpu\'))\n    if any(slot[\'reference\'].shape!=(1,size,size) for slot in slots):\n        raise ValueError(\'Training reference and Topo checkpoint resolution mismatch\')\n    from diffusers import DDIMScheduler,StableDiffusionImg2ImgPipeline\n    import diffusers\n    kwargs=dict(torch_dtype=torch.float16,use_safetensors=True)\n    if revision:kwargs[\'revision\']=revision\n    pipe=StableDiffusionImg2ImgPipeline.from_pretrained(model_id,**kwargs)\n    pipe.scheduler=DDIMScheduler.from_config(pipe.scheduler.config,clip_sample=False,thresholding=False)\n    pipe.to(\'cuda\');pipe.enable_attention_slicing()\n    for module in (pipe.unet,pipe.vae,pipe.text_encoder):module.eval().requires_grad_(False)\n    device=pipe._execution_device\n    prompt=(\'a typography sheet with 4 Korean Hangul glyphs arranged in 2 rows and 2 columns, \'\n            \'black ink on plain white background, flat front view, no grid lines, \'\n            \'preserve the glyphs and layout of the input image\')\n    if prompt_text:prompt=prompt_text\n    negative=\'photograph, scenery, paper texture, shadows, colored background, grid lines, decorations\'\n    with torch.no_grad():\n        positive,negative_embeds=pipe.encode_prompt(prompt,device,1,True,negative_prompt=negative)\n        embeddings=torch.cat([negative_embeds,positive])\n        pipe.scheduler.set_timesteps(steps,device=device)\n        if init_mode==\'noise\':\n            timesteps=pipe.scheduler.timesteps;effective_steps=len(timesteps)\n            initial=torch.randn((1,pipe.unet.config.in_channels,\n                canvas_size//pipe.vae_scale_factor,canvas_size//pipe.vae_scale_factor),\n                generator=torch.Generator(\'cpu\').manual_seed(seed),dtype=embeddings.dtype).to(device)\n            initial=initial*pipe.scheduler.init_noise_sigma\n        else:\n            starting_image=initial_image\n            if init_mode==\'external\':\n                with Image.open(input_sheet) as source:\n                    if source.size!=initial_image.size:raise ValueError(\'External sheet must match canvas size\')\n                    starting_image=source.convert(\'RGB\')\n            timesteps,effective_steps=pipe.get_timesteps(steps,strength,device)\n            initial=pipe.prepare_latents(pipe.image_processor.preprocess(starting_image),timesteps[:1],\n                1,1,embeddings.dtype,device,torch.Generator(\'cpu\').manual_seed(seed))\n    # Encoding stays identical to ordinary fp16 img2img. Decode BOTH runs in fp32\n    # for stable VAE/Topo input gradients; compare these paired runs, not old PNGs.\n    pipe.vae.to(dtype=torch.float32)\n    # The descriptor path does not need the legacy unbranched-skeleton metric.\n    # Keep its validation intact for modes that actually use it.\n    objective=None;calibration=[]\n    reference_rgb=torch.from_numpy(np.asarray(initial_image,np.float32).copy()/255).permute(2,0,1)[None].to(device)\n    if guidance_signal in (\'template_structure\',\'localized_extra\'):\n        from stroke_model.template_structure_guidance import TemplateStructureGuidance\n        from stroke_model.source_ink_guard import SourceInkGuard\n        guiding_objective=(objective_override if guidance_signal==\'localized_extra\' else\n            TemplateStructureGuidance(model,slots)).to(device).eval()\n        source_guard=SourceInkGuard(reference_rgb).to(device)\n        if preservation_weight:\n            from stroke_model.template_structure_guidance import PreserveGeneratedImage\n            anchor=torch.from_numpy(np.asarray(starting_image,np.float32).copy()/255).permute(2,0,1)[None].to(device)\n            guiding_objective=PreserveGeneratedImage(guiding_objective,anchor,preservation_weight).to(device)\n    else:\n        objective=TopoGuidance(model,slots,structure,extra_weight=extra_weight,**focus,\n            appearance_weight=appearance_weight,appearance_config=appearance,mask_gradient=mask_gradient,\n            source_spill_weight=source_spill_weight,direct_reference=direct_reference).to(device).eval()\n        if appearance_target==\'extractor_reference\':\n            calibration=objective.calibrate_appearance(reference_rgb)\n        else:objective.set_reference_sheet(reference_rgb)\n        source_guard=objective.source_guard\n        guiding_objective=objective\n    del reference_rgb\n    destination.mkdir(parents=True,exist_ok=False)\n    if init_mode==\'external\':starting_image.save(destination/\'external_input.png\')\n    checkpoint_hash=_file_hash(checkpoint)\n    initial_hash=tensor_hash(initial)\n    report=dict(guidance_signal=guidance_signal,init_mode=init_mode,baseline_only=bool(baseline_only),\n        preservation_weight=preservation_weight,preservation_anchor="external_generated_input" if preservation_weight else None,\n        reference_encoded_as_input=init_mode==\'reference\',\n        external_input=str(input_sheet) if input_sheet else None,\n        pixel_audit_is_auxiliary=guidance_signal in (\'template_structure\',\'localized_extra\'),\n        experiment=\'paired_joint_topo_backward_guidance\',status=\'running\',model_id=model_id,\n        requested_revision=revision,checkpoint=str(checkpoint),checkpoint_sha256=checkpoint_hash,\n        checkpoint_epoch=epoch,training_manifest=str(manifest),selected_manifest_rows=selected,\n        seed=seed,strength=strength,glyph_count=4,steps=steps,effective_steps=int(effective_steps),\n        timesteps=[int(t) for t in timesteps],initial_latent_sha256=initial_hash,\n        guidance_config=asdict(config),structure_config=asdict(structure),extra_weight=extra_weight,\n        stroke_focus=focus,compare_uniform=bool(compare_uniform),\n        appearance_weight=appearance_weight,appearance_config=asdict(appearance),compare_structure=bool(compare_structure),\n        appearance_target=appearance_target,reference_calibration=calibration,\n        mask_gradient=mask_gradient,source_spill_weight=source_spill_weight,direct_reference=direct_reference,\n        source_ink_guard=dict(support_radius=1,patch_radius=2),\n        primary_signal=\'frozen Topo soft-mask missing/split/junction\',\n        primary_score=\'StructuralIndicator on extracted masks with extra_weight=0\',\n        gradient_path=(\'clean latent -> fp32 VAE -> ink with detached Topo ownership -> stroke loss\'\n                       if mask_gradient==\'ink_only\' else\n                       \'clean latent -> fp32 VAE -> four torch crops -> frozen Topo -> stroke loss\'),\n        ownership_backward=mask_gradient==\'full\',\n        editing_mode=\'local_reference\' if config.local_edit else \'topo_gradient\',\n        local_edit_limits=dict(budget_ratio=.08,protect_weight=20.,max_protected_rms=.03,max_protected_peak=.12),\n        unet_backward=False,weights_updated=False,denoise_eta=0.,cfg_scale=7.5,\n        vae_decode_dtype=\'float32\',ddim_state_dtype=\'float32\',\n        unet_dtype=str(next(pipe.unet.parameters()).dtype).replace(\'torch.\',\'\'),\n        diffusers_version=diffusers.__version__,torch_version=torch.__version__,\n        prompt=prompt,negative_prompt=negative,scheduler=dict(pipe.scheduler.config),pages=[],results=[])\n\n    if guidance_signal==\'template_structure\':\n        report.update(editing_mode=\'template_normalized_geometry\',primary_signal=\'frozen extractor normalized geometry descriptors\',\n            primary_score=\'descriptor loss (not calibrated quality)\',ownership_backward=False,\n            gradient_path=\'clean latent -> VAE -> ink with detached extractor ownership -> normalized descriptors\',\n            strength=None if init_mode==\'noise\' else strength)\n\n    if guidance_signal==\'localized_extra\':\n        report.update(editing_mode=\'fixed_detected_extra_map\',primary_signal=\'predicted extra-ink mask; background from generated input\',\n            primary_score=\'fixed-region edit loss\',ownership_backward=False,\n            gradient_path=\'clean latent -> VAE -> fixed predicted-mask loss; no template similarity\',\n            preservation_anchor=\'first_generated_input\')\n\n    def write_report():\n        (destination/\'generation_report.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n\n    baseline_source_map=None\n    def save_page(image,method,filtered=None):\n        nonlocal baseline_source_map\n        page_id=len(report[\'pages\']);path=destination/f\'page_{page_id:03d}_{method}.png\';image.save(path)\n        label=f\'{method} | 4 glyphs | \'+(\'noise initialization\' if init_mode==\'noise\' else f\'strength={strength}\')+f\' seed={seed}\'\n        page=dict(page_id=page_id,method=method,label=label,input=str(path),glyph_count=4,\n                  control=method==\'reference\',guidance_on=method!=\'baseline\' and method!=\'reference\',initial_latent_sha256=initial_hash,\n                  appearance_weight=appearance_weight if method==\'guided\' else 0.)\n        report[\'pages\'].append(page)\n        rgb=torch.from_numpy(np.asarray(image,np.float32).copy()/255).permute(2,0,1)[None].to(device)\n        with torch.no_grad():\n            source=source_guard(rgb)\n            if guidance_signal in (\'template_structure\',\'localized_extra\'):\n                value,detail=guiding_objective(rgb)\n                report.setdefault(\'structure_audit\',{})[method]=dict(loss=float(value),\n                    per_glyph=[float(r.get(\'descriptor_loss\',r.get(\'edit_loss\',0.))) for r in detail[\'per_glyph\']],\n                    structure_loss=float(detail.get(\'structure_loss\',value)),\n                    preservation_loss=float(detail.get(\'preservation_loss\',0.)))\n        source_map=source[\'local_excess\'].detach().cpu()\n        audit=dict(source_spill_loss=float(source[\'loss\']))\n        if method==\'baseline\':baseline_source_map=source_map\n        elif method!=\'reference\':\n            increase=float((source_map-baseline_source_map).amax().clamp_min(0))\n            audit.update(new_local_ink_vs_baseline=increase,\n                passes_final_ink_check=increase<=(config.max_new_local_ink if config.max_new_local_ink is not None else .002)+1e-6)\n            if method==\'guided\' and guidance_signal==\'pixel_control\':report[\'quality_review_required\']=not audit[\'passes_final_ink_check\']\n        report.setdefault(\'source_ink_audit\',{})[method]=audit\n        for slot in slots:\n            prefix=destination/f\'sample_{len(report["results"]):04d}\'\n            crop=image.crop(tuple(slot[\'bbox\']));crop.save(str(prefix)+\'_raw.png\')\n            ink=scored_ink(crop,size)\n            np.savez_compressed(str(prefix)+\'.npz\',image=ink,reference=slot[\'reference\'],reference_strokes=slot[\'strokes\'])\n            report[\'results\'].append(dict(**slot[\'entry\'],input=str(prefix)+\'_raw.png\',npz=str(prefix)+\'.npz\',\n                page_id=page_id,page_input=str(path),glyph_count=4,slot=slot[\'slot\'],crop_bbox=slot[\'bbox\'],\n                method=method,guidance_on=method not in (\'baseline\',\'reference\'),control=method==\'reference\',strength=None if init_mode==\'noise\' else strength,seed=seed,\n                safety_filtered=filtered,label=label+f\' | slot={slot["slot"]+1}\'))\n        write_report()\n\n    def callback(method):\n        def step(record,before,after):\n            editor=getattr(objective,\'local_editor\',None)\n            if editor is not None and not (destination/\'local_edit_maps.npz\').exists():\n                maps={name:getattr(editor,name).detach().cpu().numpy() for name in (\'erase\',\'fill\',\'protect\')}\n                np.savez_compressed(destination/\'local_edit_maps.npz\',**maps)\n                for name,values in maps.items():\n                    Image.fromarray((values[0,0]*255).astype(np.uint8)).save(destination/f\'local_{name}.png\')\n            print(f"{method} {record[\'step\']+1}/{len(timesteps)} | {format_guidance_record(record)}",flush=True)\n            with (destination/f\'trace_{method}.jsonl\').open(\'a\',encoding=\'utf-8\') as stream:\n                stream.write(json.dumps(record)+\'\\n\')\n            if record[\'step\']%config.snapshot_every==0 or record[\'step\']==len(timesteps)-1:\n                with torch.no_grad():\n                    for tag,value in [(\'before\',before),(\'after\',after)] if record[\'accepted\'] else [(\'estimate\',after)]:\n                        rgb=decode_estimate(pipe,value).clamp(0,1)[0].permute(1,2,0).cpu().numpy()\n                        Image.fromarray(np.rint(rgb*255).astype(np.uint8)).save(\n                            destination/f\'{method}_step_{record["step"]:02d}_{tag}.png\')\n        return step\n\n    def finish(latents):\n        with torch.no_grad():\n            decoded=pipe.vae.decode(latents.float()/pipe.vae.config.scaling_factor,return_dict=False)[0]\n            checked,flags=pipe.run_safety_checker(decoded,device,embeddings.dtype)\n            images=pipe.image_processor.postprocess(checked,output_type=\'pil\',\n                do_denormalize=[not flags[0]] if flags is not None else [True])\n        return images[0],bool(flags[0]) if flags is not None else None\n\n    try:\n        save_page(initial_image,\'reference\')\n        # Both receive copies of the same VAE-sampled, noised sheet; no re-seeding shortcut.\n        methods=[(\'baseline\',GuidanceConfig(noise_ratio=0))]\n        if compare_uniform and not baseline_only:methods.append((\'uniform_guided\',config))\n        if compare_structure and appearance_weight>0 and not baseline_only:methods.append((\'structure_guided\',config))\n        if not baseline_only:methods.append((\'guided\',config))\n        for method,cfg in methods:\n            if objective is not None:\n                objective.focus_strength=0. if method==\'uniform_guided\' else focus[\'focus_strength\']\n                objective.appearance_weight=appearance_weight if method==\'guided\' else 0.\n            latents,trace=denoise(pipe,initial,timesteps,embeddings,objective=guiding_objective,config=cfg,on_step=callback(method))\n            report[method+\'_final_latent_sha256\']=tensor_hash(latents)\n            if method==\'baseline\':baseline_final=latents.detach().float().cpu()\n            else:\n                delta=float((latents.detach().float().cpu()-baseline_final).square().mean().sqrt())\n                report[method+\'_final_latent_rms_difference\']=delta\n                if method==\'guided\':report[\'final_latent_rms_difference\']=delta\n            image,filtered=finish(latents);save_page(image,method,filtered)\n            report[method+\'_accepted_steps\']=sum(row[\'accepted\'] for row in trace)\n        if tensor_hash(initial)!=initial_hash:raise RuntimeError(\'Initial latent changed\')\n        if _file_hash(checkpoint)!=checkpoint_hash:raise RuntimeError(\'Topo checkpoint changed\')\n        report[\'status\']=\'complete\';write_report()\n    except Exception as exc:\n        report.update(status=\'failed\',error=repr(exc));write_report();raise\n    finally:\n        del guiding_objective,source_guard,objective,model,pipe\n        gc.collect();torch.cuda.empty_cache()\n    return report\n\n\ndef evaluate_topo_pair(report_path,checkpoint,*,show=True):\n    """Primary numerical evaluation is based on predicted masks, not source ink."""\n    import pandas as pd\n    import matplotlib.pyplot as plt\n    report=json.loads(Path(report_path).read_text(encoding=\'utf-8\'))\n    if report.get(\'status\')!=\'complete\':raise ValueError(\'Paired generation is not complete\')\n    if _file_hash(checkpoint)!=report[\'checkpoint_sha256\']:raise ValueError(\'Use the same frozen Topo checkpoint\')\n    slots=[]\n    for sample in report[\'results\'][:4]:\n        with np.load(sample[\'npz\']) as data:\n            slots.append(dict(bbox=sample[\'crop_bbox\'],reference=data[\'reference\'].copy(),strokes=data[\'reference_strokes\'].copy()))\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model,size,epoch=load_topology(checkpoint,device)\n    objective=TopoGuidance(model,slots,StructureConfig(**report[\'structure_config\']),report[\'extra_weight\'],\n        **report.get(\'stroke_focus\',{}),appearance_weight=report.get(\'appearance_weight\',0.),\n        appearance_config=AppearanceConfig(**report.get(\'appearance_config\',{})),\n        mask_gradient=report.get(\'mask_gradient\',\'full\'),source_spill_weight=report.get(\'source_spill_weight\',0.),\n        direct_reference=report.get(\'direct_reference\',False)).to(device).eval()\n    if report.get(\'appearance_target\',\'partition_reference\')==\'extractor_reference\':\n        reference_page=next(p for p in report[\'pages\'] if p[\'method\']==\'reference\')\n        with Image.open(reference_page[\'input\']) as image:\n            reference_rgb=torch.from_numpy(np.asarray(image.convert(\'RGB\'),np.float32).copy()/255).permute(2,0,1)[None].to(device)\n        objective.calibrate_appearance(reference_rgb)\n        del reference_rgb\n    elif \'source_ink_guard\' in report:\n        reference_page=next(p for p in report[\'pages\'] if p[\'method\']==\'reference\')\n        with Image.open(reference_page[\'input\']) as image:\n            reference_rgb=torch.from_numpy(np.asarray(image.convert(\'RGB\'),np.float32).copy()/255).permute(2,0,1)[None].to(device)\n        objective.set_reference_sheet(reference_rgb)\n        del reference_rgb\n    out=Path(report_path).parent/(\'topo_evaluation_\'+datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\'))\n    out.mkdir(exist_ok=False);rows=[];stroke_rows=[];snapshots=[]\n    for page in report[\'pages\']:\n        with Image.open(page[\'input\']) as image:\n            rgb=torch.from_numpy(np.asarray(image.convert(\'RGB\'),np.float32).copy()/255).permute(2,0,1)[None].to(device)\n        objective.focus_strength=(0. if page[\'method\']==\'uniform_guided\' else report.get(\'stroke_focus\',{}).get(\'focus_strength\',0.))\n        objective.appearance_weight=report.get(\'appearance_weight\',0.) if page[\'method\']==\'guided\' else 0.\n        rgb.requires_grad_(True)\n        with torch.enable_grad():loss,details=objective(rgb,return_masks=True)\n        for values in details[\'per_glyph\']:\n            i=values[\'slot\'];slot=slots[i]\n            row={key:float(value.detach()) if torch.is_tensor(value) else value for key,value in values.items()\n                 if key not in (\'image\',\'soft_masks\',\'probabilities\',\'per_stroke\',\'defect_maps\',\'appearance_maps\')}\n            row.update(method=page[\'method\'],sheet_loss=float(loss.detach()),primary_score=\'extracted stroke score, extra excluded\')\n            row[\'slot\']=i+1  # Human-readable table; crop metadata remains zero-based.\n            rows.append(row)\n            arrays={key:values[value][0].detach().cpu().numpy() for key,value in\n                    [(\'image\',\'image\'),(\'soft_masks\',\'soft_masks\'),(\'probabilities\',\'probabilities\')]}\n            arrays.update(reference=slot[\'reference\'],reference_strokes=slot[\'strokes\'])\n            arrays[\'appearance_reference_masks\']=objective.appearance_metrics[i].reference[0].detach().cpu().numpy()\n            gradients=[]\n            for stroke in values[\'per_stroke\']:\n                stroke_row={key:float(value.detach()) if torch.is_tensor(value) else value for key,value in stroke.items()}\n                stroke_row.update(method=page[\'method\'],slot=i+1)\n                stroke_rows.append(stroke_row)\n                # Input-RGB derivative of THIS weighted stroke contribution; not a latent gradient.\n                with torch.enable_grad():gradient,=torch.autograd.grad(stroke[\'contribution\'],rgb,retain_graph=True)\n                x0,y0,x1,y1=slot[\'bbox\']\n                gradient=torch.nn.functional.interpolate(gradient[...,y0:y1,x0:x1],\n                    slot[\'reference\'].shape[-2:],mode=\'bilinear\',align_corners=False)\n                gradients.append(gradient[0].mean(0).detach().cpu().numpy())\n            arrays[\'stroke_rgb_gradients\']=np.stack(gradients)\n            for name in (\'missing\',\'split\',\'junction\'):\n                arrays[name+\'_maps\']=np.stack([m[name][0,0].cpu().numpy() for m in values[\'defect_maps\']])\n            for name,value in values[\'appearance_maps\'].items():arrays[name]=value[0].cpu().numpy()\n            prefix=out/f\'{page["method"]}_slot_{i+1}\'\n            np.savez_compressed(str(prefix)+\'.npz\',**arrays)\n            sample=next(s for s in report[\'results\'] if s[\'page_id\']==page[\'page_id\'] and s[\'slot\']==i)\n            snapshots.append({**sample,\'npz\':str(prefix)+\'.npz\'})\n            masks=arrays[\'soft_masks\'];hard=export_hard_masks(arrays[\'image\'],masks,\n                output_mode=getattr(model,\'output_mode\',\'partition\'))\n            fig,axes=plt.subplots(8,len(masks)+1,figsize=(2*(len(masks)+1),17),squeeze=False)\n            for ax in axes.flat:ax.axis(\'off\')\n            axes[0,0].imshow(arrays[\'image\'][0],cmap=\'gray_r\',vmin=0,vmax=1)\n            axes[1,0].imshow(slot[\'reference\'][0],cmap=\'gray_r\',vmin=0,vmax=1)\n            axes[0,0].set_title(\'Generated crop\');axes[1,0].set_title(\'Reference\')\n            for ax,label in zip(axes[2:,0],(\'Missing map\',\'Split map\',\'Junction map\',\'Contour discrepancy\',\n                \'Local ink / thickness discrepancy\',\'RGB loss gradient\\nred: darken / blue: brighten\')):\n                ax.text(.5,.5,label,ha=\'center\',va=\'center\',transform=ax.transAxes)\n            for k in range(len(masks)):\n                s=values[\'per_stroke\'][k]\n                axes[0,k+1].imshow(masks[k],cmap=\'gray_r\',vmin=0,vmax=1)\n                axes[0,k+1].set_title(f"Stroke {k+1} | {float(s[\'score\'].detach()):.1f}\\nw={float(s[\'weight\']):.2f}, share={float(s[\'weight_share\']):.0%}")\n                axes[1,k+1].imshow(hard[k],cmap=\'gray_r\',vmin=0,vmax=1);axes[1,k+1].set_title(f\'Export {k+1}\')\n                for axis,name in zip(axes[2:5,k+1],(\'missing\',\'split\',\'junction\')):\n                    axis.imshow(arrays[name+\'_maps\'][k],cmap=\'magma\',vmin=0)\n                for axis,name in zip(axes[5:7,k+1],(\'contour_map\',\'local_mass_map\')):\n                    axis.imshow(arrays[name][k],cmap=\'magma\',vmin=0)\n                g=arrays[\'stroke_rgb_gradients\'][k];limit=max(float(abs(g).max()),1e-12)\n                axes[7,k+1].imshow(g,cmap=\'bwr\',vmin=-limit,vmax=limit)\n            fig.suptitle(f"{page[\'method\']} | slot {i+1} | TOPO extracted-stroke score={row[\'extracted_stroke_score_no_extra\']:.1f}")\n            fig.tight_layout();fig.savefig(str(prefix)+\'.png\',dpi=140)\n            if show:plt.show()\n            plt.close(fig)\n        del loss,details,rgb,values,stroke,s\n    frame=pd.DataFrame(rows);frame.to_csv(out/\'topo_scores.csv\',index=False,encoding=\'utf-8-sig\')\n    if report.get(\'source_ink_audit\'):\n        pd.DataFrame.from_dict(report[\'source_ink_audit\'],orient=\'index\').rename_axis(\'method\').to_csv(\n            out/\'source_ink_checks.csv\',encoding=\'utf-8-sig\')\n    stroke_frame=pd.DataFrame(stroke_rows)\n    stroke_frame.to_csv(out/\'stroke_scores.csv\',index=False,encoding=\'utf-8-sig\')\n    paired=stroke_frame.pivot(index=[\'slot\',\'stroke\'],columns=\'method\',values=\'score\')\n    paired[\'guided_minus_baseline\']=paired[\'guided\']-paired[\'baseline\']\n    if \'uniform_guided\' in paired:paired[\'guided_minus_uniform\']=paired[\'guided\']-paired[\'uniform_guided\']\n    if \'structure_guided\' in paired:paired[\'guided_minus_structure\']=paired[\'guided\']-paired[\'structure_guided\']\n    paired.to_csv(out/\'stroke_comparison.csv\',encoding=\'utf-8-sig\')\n    appearance_paired=stroke_frame.pivot(index=[\'slot\',\'stroke\'],columns=\'method\',values=\'appearance_loss\')\n    appearance_paired[\'baseline_minus_guided\']=appearance_paired[\'baseline\']-appearance_paired[\'guided\']\n    if \'structure_guided\' in appearance_paired:\n        appearance_paired[\'structure_minus_guided\']=appearance_paired[\'structure_guided\']-appearance_paired[\'guided\']\n    appearance_paired.to_csv(out/\'appearance_comparison.csv\',encoding=\'utf-8-sig\')\n    if show:\n        from IPython.display import display\n        display(stroke_frame[[\'method\',\'slot\',\'stroke\',\'score\',\'weight\',\'weight_share\',\'missing\',\'split\',\'junction\',\n                             \'appearance_loss\',\'contour\',\'local_mass\']])\n        display(paired)\n        display(appearance_paired)\n    extraction=dict(checkpoint=str(checkpoint),checkpoint_epoch=epoch,output_mode=getattr(model,\'output_mode\',\'partition\'),results=snapshots)\n    (out/\'topo_report.json\').write_text(json.dumps(dict(rows=rows,stroke_rows=stroke_rows,extraction=extraction),ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    import zipfile\n    with zipfile.ZipFile(out/\'guidance_debug.zip\',\'w\',zipfile.ZIP_DEFLATED) as archive:\n        for file in out.iterdir():\n            if file.suffix!=\'.zip\':archive.write(file,\'topo_evaluation/\'+file.name)\n        for file in Path(report_path).parent.iterdir():\n            if file.is_file() and file.suffix in (\'.png\',\'.json\',\'.jsonl\',\'.csv\'):archive.write(file,\'generation/\'+file.name)\n    return frame,extraction,str(out/\'guidance_debug.zip\')\n', 'stroke_model/soft_assignment_guidance.py': '"""Reference-conditioned soft ownership guidance with constrained latent editing.\n\nOwnership is refreshed once per diffusion step, then detached for all inner\nupdates and line-search candidates. No union-outside gate or RGB painting.\n"""\nimport math\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass SoftAssignmentGuidance(nn.Module):\n    def __init__(self,model,slots,source_rgb,*,mode=\'softmax\',focus_slots=(0,),\n                 allowance=3.,falloff=3.,leak_floor=.05,preserve_weight=.1,\n                 healthy_rms_limit=.025,healthy_local_limit=.12,healthy_mass_drop=.04,\n                 baseline_rms_slack=.002,baseline_local_slack=.02,baseline_mass_slack=.01):\n        super().__init__()\n        if mode not in (\'softmax\',\'geometry\'):raise ValueError(\'Unknown assignment guide mode\')\n        values=(allowance,falloff,preserve_weight,healthy_rms_limit,healthy_local_limit,healthy_mass_drop,\n                baseline_rms_slack,baseline_local_slack,baseline_mass_slack)\n        if any(not math.isfinite(v) or v<0 for v in values) or falloff<=0 or not 0<=leak_floor<1:\n            raise ValueError(\'Invalid guide configuration\')\n        if not focus_slots or any(i not in range(len(slots)) for i in focus_slots):raise ValueError(\'Invalid focus slots\')\n        if source_rgb.ndim!=4 or source_rgb.shape[:2]!=(1,3):raise ValueError(\'One RGB image required\')\n        if not torch.isfinite(source_rgb).all() or (source_rgb<0).any() or (source_rgb>1).any():\n            raise ValueError(\'Source RGB must be finite and in [0,1]\')\n        self.model=model.eval().requires_grad_(False);self.slots=slots\n        self.mode=mode;self.focus_slots=tuple(focus_slots);self.leak_floor=leak_floor\n        self.preserve_weight=preserve_weight;self.healthy_rms_limit=healthy_rms_limit\n        self.healthy_local_limit=healthy_local_limit;self.healthy_mass_drop=healthy_mass_drop\n        self.baseline_rms_slack=baseline_rms_slack;self.baseline_local_slack=baseline_local_slack\n        self.baseline_mass_slack=baseline_mass_slack;self.baseline_guards={}\n        self.register_buffer(\'source\',source_rgb.detach().float().clone())\n        self.register_buffer(\'source_ink\',self.ink(source_rgb).detach())\n        healthy=[];bands=[];guard_names=[\'protected_rms\',\'protected_local_rms\']\n        target=torch.ones_like(source_rgb)\n        for i,slot in enumerate(slots):\n            reference=source_rgb.new_tensor(np.asarray(slot[\'reference\']))[None]\n            strokes=source_rgb.new_tensor(np.asarray(slot[\'strokes\']))[None]\n            distance=np.stack([distance_transform_edt(s<.1) for s in np.asarray(slot[\'strokes\'])])\n            d=source_rgb.new_tensor(distance)[None]\n            crop=self.crop(self.source_ink,slot,reference.shape[-2:])\n            # Calibrate thickness from the observed image, using nearest-stroke\n            # pixels; quantiles suppress isolated protrusions. Freeze thereafter.\n            nearest=d.argmin(1,keepdim=True)\n            radii=[]\n            for k in range(strokes.shape[1]):\n                samples=d[:,k:k+1][(crop>.5)&(nearest==k)&(d[:,k:k+1]<=6)]\n                radius=max(allowance,float(torch.quantile(samples,.85))+.5) if samples.numel() else allowance\n                radii.append(min(radius,6.))\n            radius=d.new_tensor(radii)[None,:,None,None]\n            cost=1-torch.exp(-F.relu(d-radius)/falloff)\n            self.register_buffer(f\'reference_{i}\',reference)\n            self.register_buffer(f\'strokes_{i}\',strokes)\n            self.register_buffer(f\'cost_{i}\',cost)\n            bands.append(radii)\n            x0,y0,x1,y1=slot[\'bbox\']\n            observed=source_rgb[...,y0:y1,x0:x1]\n            pixels=observed[0].permute(1,2,0).reshape(-1,3)\n            light=pixels.mean(1);bright=pixels[light>=torch.quantile(light,.8)]\n            color=bright.median(0).values\n            if color.mean()<.7 or (bright-color).abs().median()>.08:\n                raise ValueError(\'This deletion experiment needs a bright flat background\')\n            target[...,y0:y1,x0:x1]=color[None,:,None,None]\n            for k in range(strokes.shape[1]):\n                region=((d[:,k:k+1]<=radius[:,k:k+1])&(nearest==k)&(crop>.25)).float()\n                sheet=torch.zeros_like(self.source_ink)\n                sheet[...,y0:y1,x0:x1]=F.interpolate(region,(y1-y0,x1-x0),mode=\'nearest\')\n                healthy.append(sheet)\n                guard_names.append(f\'slot_{i+1}_stroke_{k+1}_mass_drop\')\n        self.bands=bands\n        self.guard_names=guard_names\n        self.register_buffer(\'target\',target.detach())\n        self.register_buffer(\'healthy\',torch.cat(healthy,1))\n        self.register_buffer(\'initial_weights\',torch.zeros_like(self.source_ink))\n        self.register_buffer(\'weights\',torch.zeros_like(self.source_ink))\n        self.register_buffer(\'gradient_support\',torch.zeros_like(self.source_ink))\n        self.register_buffer(\'protect\',torch.ones_like(self.source_ink))\n        initial=self.compute_weights(source_rgb)\n        if not (initial>.01).any():raise ValueError(\'No soft extra-ink signal; inspect reference alignment\')\n        self.initial_weights.copy_(initial);self.weights.copy_(initial)\n        self.refresh(source_rgb)\n\n    @staticmethod\n    def ink(rgb):\n        weights=rgb.new_tensor([.299,.587,.114])[None,:,None,None]\n        return (1-(rgb*weights).sum(1,keepdim=True)).clamp(0,1)\n\n    @staticmethod\n    def crop(value,slot,size):\n        x0,y0,x1,y1=slot[\'bbox\']\n        return F.interpolate(value[...,y0:y1,x0:x1],size,mode=\'bilinear\',align_corners=False)\n\n    @torch.no_grad()\n    def compute_weights(self,rgb):\n        ink=self.ink(rgb);result=torch.zeros_like(ink)\n        for i in self.focus_slots:\n            slot=self.slots[i];reference=getattr(self,f\'reference_{i}\')\n            strokes=getattr(self,f\'strokes_{i}\');cost=getattr(self,f\'cost_{i}\')\n            crop=self.crop(ink,slot,reference.shape[-2:])\n            if self.mode==\'softmax\':\n                p=self.model(crop,reference,strokes)[\'probabilities\'].float()\n                if p.shape!=cost.shape or not torch.isfinite(p).all():raise ValueError(\'Invalid extractor probabilities\')\n                if (p<0).any() or not torch.allclose(p.sum(1),torch.ones_like(p[:,0]),atol=1e-3):\n                    raise ValueError(\'A normalized stroke softmax is required\')\n                pressure=(p.detach()*cost).sum(1,keepdim=True)\n            else:pressure=cost.amin(1,keepdim=True)\n            pressure=F.relu(pressure-self.leak_floor)/(1-self.leak_floor)\n            source=self.crop(self.source_ink,slot,reference.shape[-2:])\n            selection=(.5*source+.5*crop-.05).clamp_min(0)\n            x0,y0,x1,y1=slot[\'bbox\']\n            result[...,y0:y1,x0:x1]=F.interpolate(pressure*selection,(y1-y0,x1-x0),mode=\'bilinear\',align_corners=False)\n        return result.detach()\n\n    @torch.no_grad()\n    def refresh(self,rgb):\n        # Retain pressure on initial defects so that fading residuals do not\n        # disappear from the objective as the current ink or ownership changes.\n        self.weights.copy_(torch.maximum(self.compute_weights(rgb),.25*self.initial_weights))\n        self.gradient_support.copy_(F.max_pool2d(self.weights,25,1,12).clamp(0,1).sqrt())\n        # Binary decisions are for the guard only; the editing loss stays soft.\n        edit_halo=F.max_pool2d((self.initial_weights>.03).float(),17,1,8)\n        normal=self.healthy.amax(1,keepdim=True)\n        self.protect.copy_(torch.maximum(1-edit_halo,normal))\n\n    def terms(self,rgb,*,weights=None):\n        weights=self.weights if weights is None else weights\n        difference=rgb-self.target\n        error=(difference.square()+.5*difference.abs()).mean(1,keepdim=True)\n        edit=(weights*error).sum()/weights.sum().clamp_min(1e-6)\n        change=(rgb-self.source).square().mean(1,keepdim=True)\n        keep=(change*self.protect).sum()/self.protect.sum().clamp_min(1)\n        return edit,keep\n\n    def forward(self,rgb):\n        edit,keep=self.terms(rgb)\n        rows=[]\n        for i,slot in enumerate(self.slots):\n            x0,y0,x1,y1=slot[\'bbox\'];w=self.weights[...,y0:y1,x0:x1]\n            error=(rgb-self.target).abs().mean(1,keepdim=True)[...,y0:y1,x0:x1]\n            rows.append(dict(slot=i,edit_loss=(w*error).sum()/w.sum().clamp_min(1e-6),per_stroke=[]))\n        return edit+self.preserve_weight*keep,dict(per_glyph=rows,edit_loss=edit,\n            structure_loss=edit,preservation_loss=keep)\n\n    @torch.no_grad()\n    def guard_vector(self,rgb):\n        change=(rgb-self.source).square().mean(1,keepdim=True)*self.protect\n        rms=(change.sum()/self.protect.sum().clamp_min(1)).sqrt()\n        local=F.avg_pool2d(change,5,1,2).amax().sqrt()\n        ink=self.ink(rgb)\n        original=(self.source_ink*self.healthy).sum((-2,-1))\n        current=(ink*self.healthy).sum((-2,-1))\n        drops=((original-current)/original.clamp_min(1)).clamp_min(0).flatten()\n        return torch.cat((torch.stack((rms,local)),drops))\n\n    @torch.no_grad()\n    def guard_values(self,rgb):\n        vector=self.guard_vector(rgb)\n        return torch.stack((vector[0],vector[1],vector[2:].amax()))\n\n    def begin_baseline(self):\n        self.baseline_guards.clear()\n\n    @torch.no_grad()\n    def record_baseline(self,index,rgb):\n        self.baseline_guards[int(index)]=self.guard_vector(rgb).detach().clone()\n\n    @torch.no_grad()\n    def guard_limits(self,anchor_rgb,step_index=None):\n        anchor=self.guard_vector(anchor_rgb)\n        baseline=self.baseline_guards.get(step_index,anchor)\n        floors=baseline.new_full(baseline.shape,self.healthy_mass_drop)\n        floors[0]=self.healthy_rms_limit;floors[1]=self.healthy_local_limit\n        slack=baseline.new_full(baseline.shape,self.baseline_mass_slack)\n        slack[0]=self.baseline_rms_slack;slack[1]=self.baseline_local_slack\n        # Anchor to the SAME un-guided trajectory, not to the previous accepted\n        # edit. This budget cannot accumulate anew with each inner iteration.\n        return torch.maximum(floors,baseline)+slack\n\n    def optimize_clean(self,clean,decode,*,iterations,step_ratio,backtracks,budget_ratio,step_index=None):\n        with torch.no_grad():\n            anchor=clean.detach().float();anchor_rgb=decode(anchor)\n            self.refresh(anchor_rgb)\n            limits=self.guard_limits(anchor_rgb,step_index)\n            initial_edit,initial_keep=self.terms(anchor_rgb)\n            initial_fixed_edit,_=self.terms(anchor_rgb,weights=self.initial_weights)\n            initial_guard=self.guard_vector(anchor_rgb)\n            initial_total=initial_edit+self.preserve_weight*initial_keep\n        value=anchor.clone();attempts=[];accepted_count=0\n        scale=anchor.square().mean().sqrt().clamp_min(1e-3)\n        support=F.adaptive_max_pool2d(self.gradient_support,anchor.shape[-2:])\n        for inner in range(iterations):\n            with torch.enable_grad():\n                leaf=value.detach().requires_grad_(True)\n                rgb=decode(leaf);edit,keep=self.terms(rgb)\n                loss=edit+self.preserve_weight*keep\n                gradient,=torch.autograd.grad(loss,leaf)\n            gradient=gradient*support\n            if not torch.isfinite(gradient).all() or not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite soft-assignment gradient\')\n            norm=gradient.square().mean().sqrt()\n            if norm<1e-12:break\n            direction=gradient/norm\n            accepted=False\n            for retry in range(backtracks+1):\n                candidate=value-step_ratio*(.5**retry)*scale*direction\n                displacement=candidate-anchor\n                move=displacement.square().mean().sqrt()\n                candidate=anchor+displacement*torch.clamp(budget_ratio*scale/move.clamp_min(1e-12),max=1)\n                with torch.no_grad():\n                    image=decode(candidate);after_edit,after_keep=self.terms(image)\n                    guard=self.guard_vector(image)\n                if not torch.isfinite(after_edit) or not torch.isfinite(after_keep) or not torch.isfinite(guard).all():\n                    attempts.append(dict(inner=inner,backtrack=retry,reason=\'nonfinite_candidate\'))\n                    continue\n                reason=(\'edit_not_lower\' if after_edit>=edit.detach()-1e-8 else\n                        \'healthy_guard\' if (guard>limits).any() else \'accepted\')\n                attempts.append(dict(inner=inner,backtrack=retry,edit_before=float(edit.detach()),\n                    edit_after=float(after_edit),protected_rms=float(guard[0]),\n                    protected_local_rms=float(guard[1]),healthy_mass_drop=float(guard[2:].amax()),\n                    violated_guards=(guard>limits).nonzero().flatten().cpu().tolist(),\n                    guard_failures=[dict(name=self.guard_names[j],value=float(guard[j]),limit=float(limits[j]),\n                        excess=float(guard[j]-limits[j])) for j in (guard>limits).nonzero().flatten().cpu().tolist()],reason=reason))\n                if reason==\'accepted\':\n                    value=candidate.detach();accepted_count+=1;accepted=True;break\n            if not accepted:break\n        with torch.no_grad():\n            final_rgb=decode(value);final_edit,final_keep=self.terms(final_rgb)\n            final_guard=self.guard_values(final_rgb)\n            final_fixed_edit,_=self.terms(final_rgb,weights=self.initial_weights)\n        return value,dict(accepted=accepted_count>0,accepted_inner_steps=accepted_count,\n            loss_before=float(initial_total),loss_after=float(final_edit+self.preserve_weight*final_keep),\n            edit_loss_before=float(initial_edit),edit_loss_after=float(final_edit),\n            fixed_map_edit_before=float(initial_fixed_edit),fixed_map_edit_after=float(final_fixed_edit),\n            anchor_guard_failures=[dict(name=self.guard_names[j],value=float(initial_guard[j]),limit=float(limits[j]))\n                for j in (initial_guard>limits).nonzero().flatten().cpu().tolist()],\n            preservation_loss_before=float(initial_keep),preservation_loss_after=float(final_keep),\n            protected_rms=float(final_guard[0]),protected_local_rms=float(final_guard[1]),\n            healthy_mass_drop=float(final_guard[2]),\n            guard_limits=[float(limits[0]),float(limits[1]),float(limits[2:].amax())],\n            per_stroke_guard_limits=limits[2:].cpu().tolist(),\n            guard_reference=\'matched_unguided_step\' if step_index in self.baseline_guards else \'anchor_fallback\',\n            soft_weight_sum=float(self.weights.sum()),attempts=attempts)\n\n    @torch.no_grad()\n    def diagnostics(self,rgb):\n        edit,keep=self.terms(rgb,weights=self.initial_weights)\n        guard=self.guard_values(rgb)\n        return dict(initial_map_edit_loss=float(edit),protected_mse=float(keep),\n            protected_rms=float(guard[0]),protected_local_rms=float(guard[1]),healthy_mass_drop=float(guard[2]))\n'}
for name,code in runtime_helpers.items():
    (root/name).write_text(code,encoding='utf-8')
sys.path.insert(0,str(root))
for name in list(sys.modules):
    if name.split('.')[0] in ('stroke_model','diffusion_model'):del sys.modules[name]
import torch,numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files
if not torch.cuda.is_available():raise RuntimeError('L4 GPU 런타임을 선택하세요.')
from stroke_model.convert_reference_csv import existing_path
from diffusion_model.saved_repair_demo import prepare_saved_demo
import shutil
from diffusion_model.saved_repair_demo import pack_saved_generation
HSQM_PATH='/content/drive/Shareddrives/2026 자율연구/HSQM'
HSQM=existing_path(HSQM_PATH)
ARCHIVE=None  # Drive ZIP 경로를 명시할 수 있음. None이면 아래 위치/원래 생성 폴더 확인
CHECKPOINT=None  # None이면 당시 기록된 checkpoint 경로. 최신 모델 임의 선택 없음
SOURCE_REPORT=HSQM/'topo_guided_diffusion/20261108_103932_929716/generation_report.json'
if ARCHIVE is None:
    candidates=[HSQM/'guidance_debug (4).zip',HSQM/'debug/guidance_debug (4).zip',
        Path('/content/drive/MyDrive/guidance_debug (4).zip'),
        Path('/content/drive/MyDrive/Downloads/guidance_debug (4).zip')]
    ARCHIVE=next((p for p in candidates if p.is_file()),None)
if ARCHIVE is None:
    try:report_path=existing_path(SOURCE_REPORT)
    except FileNotFoundError:report_path=None
    if report_path is not None:
        print('Drive의 기존 생성 실험에서 입력을 복원합니다:',report_path,flush=True)
        try:ARCHIVE=pack_saved_generation(report_path,root/'source_experiment.zip')
        except FileNotFoundError as exc:print('기존 생성 폴더의 파일이 부족합니다:',exc)
if ARCHIVE is None:
    print('기존 ZIP/생성 폴더를 찾지 못했습니다. guidance_debug (4).zip만 업로드하세요.')
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError('ZIP 하나를 선택하세요.')
    ARCHIVE=Path(next(iter(uploaded)))
archive_origin=str(ARCHIVE)
local_archive=root/'saved_input.zip'
shutil.copyfile(existing_path(ARCHIVE),local_archive)
RUN_DIR=Path('/content/soft_assignment_runs')/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
demo=prepare_saved_demo(local_archive,RUN_DIR/'input')
if CHECKPOINT is None:
    try:CHECKPOINT=existing_path(demo['checkpoint'])
    except FileNotFoundError:
        CHECKPOINT=HSQM/'v2_topology_runs/20261104_214705_208224/best.pt'
checkpoint_origin=existing_path(CHECKPOINT)
CHECKPOINT=root/'extractor_best.pt'
print('Drive 획 모델을 실행용 임시 저장소로 복사합니다:',checkpoint_origin,flush=True)
shutil.copyfile(checkpoint_origin,CHECKPOINT)
actual_sha=hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
if demo['checkpoint_sha256'] and actual_sha!=demo['checkpoint_sha256']:
    raise ValueError('ZIP 당시 획 모델과 SHA256이 다릅니다. CHECKPOINT를 확인하세요.')
(RUN_DIR/'drive_sources.json').write_text(json.dumps(dict(archive=archive_origin,checkpoint=str(checkpoint_origin)),ensure_ascii=False,indent=2))
print('GPU:',torch.cuda.get_device_name(0),'| 실제 입력:',demo['source'])
print('획 모델:',checkpoint_origin,'| SHA256:',actual_sha)
display(Image.open(demo['source']))

In [ ]:
from diffusion_model.score_pilot import training_catalog
from diffusion_model.multiglyph_pilot import compose_sheet
from stroke_model.infer_topology import load_topology
from stroke_model.soft_assignment_guidance import SoftAssignmentGuidance

RUN_VERSION='soft-assignment-v2-matched-baseline-all-slots'
FOCUS_SLOTS=(0,1,2,3)  # 네 칸 모두. 한 칸만 시험하려면 예: (0,)
ALLOWANCE=3.0     # 모델 좌표의 최소 허용 반폭. 실제 소스 두께로 상향 보정 후 고정
FALLOFF=3.0       # 거리 페널티가 완만하게 증가하는 폭
extractor,size,epoch=load_topology(CHECKPOINT,torch.device('cuda'))
extractor.requires_grad_(False)
_,slots=compose_sheet(training_catalog(demo['manifest']),demo['size'])
for slot,box in zip(slots,demo['boxes']):slot['bbox']=box
if any(slot['reference'].shape[-1]!=size for slot in slots):raise ValueError('모델/기준 해상도 불일치')
with Image.open(demo['source']) as image:array=np.asarray(image.convert('RGB'),np.float32)/255
source_rgb=torch.from_numpy(array.copy()).permute(2,0,1)[None].cuda()
options=dict(focus_slots=FOCUS_SLOTS,allowance=ALLOWANCE,falloff=FALLOFF,
    preserve_weight=.1,healthy_rms_limit=.025,healthy_local_limit=.12,healthy_mass_drop=.04,
    baseline_rms_slack=.002,baseline_local_slack=.02,baseline_mass_slack=.01)
soft_objective=SoftAssignmentGuidance(extractor,slots,source_rgb,mode='softmax',**options)
geometry_objective=SoftAssignmentGuidance(extractor,slots,source_rgb,mode='geometry',**options)
np.savez_compressed(RUN_DIR/'initial_guides.npz',softmax_weights=soft_objective.initial_weights.cpu().numpy(),
    geometry_weights=geometry_objective.initial_weights.cpu().numpy(),healthy_regions=soft_objective.healthy.cpu().numpy())
guide_info=dict(version=RUN_VERSION,kind='dynamic_detached_softmax_stroke_distance',uses_reference_conditions=True,
    options=options,calibrated_radii=soft_objective.bands,checkpoint_sha256=actual_sha,
    geometry_mean_absolute_map_difference=float((soft_objective.initial_weights-geometry_objective.initial_weights).abs().mean()))
(RUN_DIR/'guide_info.json').write_text(json.dumps(guide_info,indent=2))
fig,axes=plt.subplots(1,4,figsize=(17,4))
axes[0].imshow(array);axes[0].set_title('Saved erroneous input')
axes[1].imshow(soft_objective.initial_weights[0,0].cpu(),cmap='magma',vmin=0,vmax=1);axes[1].set_title('Softmax-weighted distance')
axes[2].imshow(geometry_objective.initial_weights[0,0].cpu(),cmap='magma',vmin=0,vmax=1);axes[2].set_title('Nearest-stroke distance only')
axes[3].imshow(array);axes[3].imshow(soft_objective.initial_weights[0,0].cpu(),cmap='Reds',alpha=.6,vmin=0,vmax=1);axes[3].set_title('Continuous edit pressure')
for ax in axes:ax.axis('off')
fig.tight_layout();fig.savefig(RUN_DIR/'guide_preview.png',dpi=140);plt.show()
for index in FOCUS_SLOTS:
    slot=slots[index];ref=getattr(soft_objective,f'reference_{index}');strokes=getattr(soft_objective,f'strokes_{index}')
    ink=soft_objective.crop(soft_objective.source_ink,slot,ref.shape[-2:])
    with torch.no_grad():prediction=extractor(ink,ref,strokes)
    p=prediction['probabilities'][0].cpu();m=prediction['masks'][0].cpu()
    fig,axes=plt.subplots(3,len(p),figsize=(3*len(p),8),squeeze=False)
    for k in range(len(p)):
        axes[0,k].imshow(strokes[0,k].cpu(),cmap='gray');axes[0,k].set_title(f'Condition {k+1}')
        axes[1,k].imshow(p[k],cmap='viridis',vmin=0,vmax=1);axes[1,k].set_title(f'Softmax p{k+1}')
        axes[2,k].imshow(m[k],cmap='magma',vmin=0,vmax=1);axes[2,k].set_title(f'Extracted ink {k+1}')
    for ax in axes.flat:ax.axis('off')
    plt.show()
    np.savez_compressed(RUN_DIR/f'extracted_slot_{index+1}.npz',probabilities=p.numpy(),masks=m.numpy())
print('보정은 위 연속값 전체를 사용합니다. 정상 획 전체에 압력이 강하면 ALLOWANCE/조건 정렬을 확인하세요.')
print('허용 반폭:',soft_objective.bands)
print('실제 보정 슬롯:',FOCUS_SLOTS)

In [ ]:
from diffusion_model.run_topo_guidance import generate_guided_pair
from huggingface_hub import model_info
import pandas as pd
import zipfile

STEPS=100
STRENGTH=.35
SEED=101
RUN_GEOMETRY_CONTROL=True
MODEL_ID='stable-diffusion-v1-5/stable-diffusion-v1-5'
REVISION=model_info(MODEL_ID).sha
GUIDANCE=dict(local_edit=False,clean_step_ratio=.01,start_fraction=.1,end_fraction=1.,
    every=1,backtracks=5,snapshot_every=2,state_update='clean_residual',max_new_local_ink=None,
    inner_iterations=4,final_inner_iterations=24,max_clean_change_ratio=.08)
if hashlib.sha256(Path(demo['source']).read_bytes()).hexdigest()!=demo['provenance']['source_sha256']:
    raise RuntimeError('입력이 바뀌었습니다. 1번 셀부터 다시 실행하세요.')
def run_pair(objective,name):
    # Reset from the same source before each method, including guard masks.
    objective.refresh(source_rgb)
    directory=RUN_DIR/name
    report=generate_guided_pair(demo['manifest'],demo['rows'],CHECKPOINT,directory,
        model_id=MODEL_ID,revision=REVISION,steps=STEPS,canvas_size=demo['size'],
        strength=STRENGTH,seed=SEED,init_mode='external',input_sheet=demo['source'],
        guidance=GUIDANCE,guidance_signal='localized_extra',objective_override=objective,
        slot_bboxes=demo['boxes'],prompt_text='Black Korean lettering on a plain white background. Preserve the existing glyphs and layout.',
        compare_structure=False,compare_uniform=False,appearance_weight=0,extra_weight=0,
        source_spill_weight=0,direct_reference=False)
    report.update(version=RUN_VERSION,primary_signal='dynamic_'+objective.mode+'_stroke_distance',
        editing_mode='iterative_clean_latent_with_separate_edit_and_healthy_guards',
        primary_score='continuous edit loss; not calibrated image quality',
        gradient_path='clean latent -> VAE -> RGB; detached extractor ownership refreshed per diffusion step',
        guide_uses_reference_conditions=True,ownership_backward=False,
        focus_slots=list(objective.focus_slots),normal_guard_reference='matched_unguided_trajectory',
        baseline_guard_steps=len(objective.baseline_guards))
    (directory/'generation_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
    np.savez_compressed(directory/'last_guidance.npz',weights=objective.weights.cpu().numpy(),protect=objective.protect.cpu().numpy())
    return report
generation=run_pair(soft_objective,'softmax_guided')
pages={p['method']:p['input'] for p in generation['pages']}
panels=[('Input',demo['source']),('Re-denoise only',pages['baseline']),('Softmax-guided',pages['guided'])]
if RUN_GEOMETRY_CONTROL:
    geometry_generation=run_pair(geometry_objective,'geometry_control')
    if generation['initial_latent_sha256']!=geometry_generation['initial_latent_sha256']:
        raise RuntimeError('대조군의 초기 잠재값이 다릅니다.')
    gp={p['method']:p['input'] for p in geometry_generation['pages']}
    if Path(pages['baseline']).read_bytes()!=Path(gp['baseline']).read_bytes():raise RuntimeError('Baseline 재현 실패')
    panels.append(('Distance-only guided',gp['guided']))
fig,axes=plt.subplots(1,len(panels),figsize=(5*len(panels),5));metrics=[]
# Common, fixed source diagnostic maps for ALL methods. Do not compare each
# method on its own changing guide. These are still optimization-related metrics.
soft_objective.refresh(source_rgb)
geometry_objective.refresh(source_rgb)
evaluation_region=(geometry_objective.initial_weights>.03).float()
for ax,(label,path) in zip(axes,panels):
    with Image.open(path) as image:rgb=np.asarray(image.convert('RGB'),np.float32)/255
    ax.imshow(rgb);ax.set_title(label);ax.axis('off')
    tensor=torch.from_numpy(rgb.copy()).permute(2,0,1)[None].cuda()
    metric=soft_objective.diagnostics(tensor)
    metric['initial_geometry_edit_loss']=geometry_objective.diagnostics(tensor)['initial_map_edit_loss']
    metric['fixed_error_region_mean_ink']=float((soft_objective.ink(tensor)*evaluation_region).sum()/evaluation_region.sum().clamp_min(1))
    metrics.append(dict(method=label,**metric))
fig.tight_layout();fig.savefig(RUN_DIR/'comparison.png',dpi=150);plt.show()
fig,axes=plt.subplots(len(FOCUS_SLOTS),len(panels),figsize=(4*len(panels),4*len(FOCUS_SLOTS)),squeeze=False)
for row,index in enumerate(FOCUS_SLOTS):
    box=slots[index]['bbox']
    for ax,(label,path) in zip(axes[row],panels):
        with Image.open(path) as image:ax.imshow(image.crop(tuple(box)))
        ax.set_title(f'Slot {index+1}: {label}');ax.axis('off')
fig.tight_layout();fig.savefig(RUN_DIR/'focused_comparison.png',dpi=160);plt.show()
frame=pd.DataFrame(metrics);display(frame);frame.to_csv(RUN_DIR/'region_diagnostics.csv',index=False)
trace=[json.loads(line) for line in (RUN_DIR/'softmax_guided/trace_guided.jsonl').read_text().splitlines()]
print('채택한 보정 단계:',generation['guided_accepted_steps'])
print('채택한 내부 수정:',sum(r.get('accepted_inner_steps',0) for r in trace))
print('삭제 손실/정상 획 한도는 trace_guided.jsonl에서 각 시도별로 확인할 수 있습니다.')
display(pd.DataFrame([{k:r.get(k) for k in ('step','accepted_inner_steps','edit_loss_before',
    'edit_loss_after','protected_rms','protected_local_rms','healthy_mass_drop')} for r in trace if r['active']]))
snapshots=sorted((RUN_DIR/'softmax_guided').glob('guided_step_*.png'))
snapshots=[p for p in snapshots if p.stem.endswith(('after','estimate'))]
if snapshots:
    frames=[]
    for path in snapshots:
        with Image.open(path) as image:frames.append(image.convert('RGB').copy())
    frames[0].save(RUN_DIR/'reverse_process.gif',save_all=True,append_images=frames[1:],duration=350,loop=0)
    from IPython.display import Image as DisplayImage
    display(DisplayImage(filename=str(RUN_DIR/'reverse_process.gif')))
print('GIF는 중간 clean 예측입니다. 최종 출력과 정상 획 손상도 함께 확인하세요.')
(RUN_DIR/'runtime_helpers.json').write_text(json.dumps(runtime_helpers,ensure_ascii=False))
(RUN_DIR/'git_commit.txt').write_text(commit)
archive_path=RUN_DIR.with_suffix('.zip')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as z:
    for path in RUN_DIR.rglob('*'):
        if path.is_file():z.write(path,path.relative_to(RUN_DIR))
drive_output=HSQM/'soft_assignment_repair_results'
drive_output.mkdir(parents=True,exist_ok=True)
drive_archive=drive_output/archive_path.name
shutil.copy2(archive_path,drive_archive)
print('Drive 결과 ZIP:',drive_archive)
files.download(str(archive_path))